# Convolutional Neural Networks

Grid over **per-layer output channels** (depth = length of the list), and **kernel size** (1, 3, 5). RGB CIFAR-10 (`in_channels=3`). Each block: Conv → ReLU → MaxPool.

In [ ]:
import sys

sys.path.append("..")

import time
from itertools import product

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision import datasets
from torchvision.transforms import v2


In [ ]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")

In [ ]:
SAVE_PATH = "../data"
BATCH_SIZE = 256
MAX_ITER = 50

KERNEL_SIZES = [3, 5, 7]
OUTPUT_CHANNEL_CONFIGS: list[list[int]] = [
    [16],
    [16, 32],
    [16, 32, 64],
    [16, 32, 64, 128],
    [32, 64, 128, 256],
]
SHALLOW_OUTPUT_CHANNEL_CONFIGS: list[list[int]] = [
    [16],
    [16, 32],
    [32, 64, 128, 256],
]

INPUT_CHANNELS = 3
IMG_SIZE = 32

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
    v2.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]),
])

train_ds = datasets.CIFAR10(
    root=SAVE_PATH, train=True, download=True, transform=transform
)
test_ds = datasets.CIFAR10(
    root=SAVE_PATH, train=False, download=True, transform=transform
)


def preload_to_device(dataset: Dataset, device: str) -> TensorDataset:
    loader = DataLoader(
        dataset=dataset, batch_size=len(dataset), num_workers=6
    )
    X, y = next(iter(loader))
    return TensorDataset(X.to(device), y.to(device))


train_dl = DataLoader(
    dataset=preload_to_device(dataset=train_ds, device=device),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
test_dl = DataLoader(
    dataset=preload_to_device(dataset=test_ds, device=device),
    batch_size=BATCH_SIZE,
)

In [ ]:
def mlp_head(
    input_size: int,
    batch_norm: bool = False,
    dropout: bool = False,
) -> nn.Sequential:
    layers: list[nn.Module] = [nn.Flatten()]
    if dropout:
        layers.append(nn.Dropout(0.3))

    layers.append(nn.Linear(input_size, 128))
    if batch_norm:
        layers.append(nn.BatchNorm1d(128))
    layers.append(nn.ReLU())
    if dropout:
        layers.append(nn.Dropout())

    layers.append(nn.Linear(128, 64))
    if batch_norm:
        layers.append(nn.BatchNorm1d(64))
    layers.append(nn.ReLU())
    if dropout:
        layers.append(nn.Dropout())

    layers.append(nn.Linear(64, 10))
    return nn.Sequential(*layers)

In [ ]:
from pytorch_helpers.neuralnetwork import NeuralNetwork

def create_conv_layers_tuples(
    output_channels: list[int],
    kernel_size: int,
    *,
    batch_norm: bool = False,
) -> tuple[nn.Sequential, int]:
    convs: list[nn.Module] = []

    in_ch = INPUT_CHANNELS
    spatial = IMG_SIZE
    padding = 1

    for out_ch in output_channels:
        spatial = spatial + 2*padding - kernel_size + 1
        spatial = (spatial - 2) // 2 + 1

        block: list[nn.Module] = [
            nn.Conv2d(in_ch, out_ch, kernel_size, padding=padding),
        ]
        if batch_norm:
            block.append(nn.BatchNorm2d(out_ch))
        block.extend([nn.ReLU(), nn.MaxPool2d(2)])

        convs.append(nn.Sequential(*block))
        in_ch = out_ch

    flatten_size = output_channels[-1] * spatial * spatial
    return nn.Sequential(*convs), flatten_size


def build_conv_model(
    output_channels: list[int],
    kernel_size: int,
    batch_norm: bool = False,
    dropout: bool = False,
) -> NeuralNetwork:
    conv_layers, flat_size = create_conv_layers_tuples(
        output_channels=output_channels,
        kernel_size=kernel_size,
        batch_norm=batch_norm,
    )
    architecture = nn.Sequential(
        conv_layers,
        mlp_head(
            input_size=flat_size,
            batch_norm=batch_norm,
            dropout=dropout,
        ),
    )
    return NeuralNetwork(architecture=architecture, flatten_input=False)

In [ ]:
from helpers.training_history import TrainingHistory
from pytorch_helpers.training import TorchMulticlassClassifier

HISTORIES: dict[str, TrainingHistory] = {}


def _config_name(
    output_channels: list[int],
    kernel_size: int,
    *,
    use_adamw: bool = False,
    batch_norm: bool = False,
    dropout: bool = False,
) -> str:
    name = f"out={output_channels}, k={kernel_size}"
    extras = []
    if use_adamw:
        extras.append("adamw")
    if batch_norm:
        extras.append("bn")
    if dropout:
        extras.append("dropout")
    if extras:
        name += ", " + "+".join(extras)
    return name


def train_conv_config(
    *,
    output_channels: list[int],
    kernel_size: int,
    max_iter: int,
    use_adamw: bool = False,
    batch_norm: bool = False,
    dropout: bool = False,
) -> dict | None:

    print(
        f"\n--- out={output_channels}, k={kernel_size}, "
        f"bn={batch_norm}, dropout={dropout} ---"
    )

    model = build_conv_model(
        output_channels=output_channels,
        kernel_size=kernel_size,
        batch_norm=batch_norm,
        dropout=dropout,
    )

    if use_adamw:
        optimizer = torch.optim.AdamW(params=model.parameters())
    else:
        optimizer = torch.optim.Adam(params=model.parameters())

    clf = TorchMulticlassClassifier(
        model=model,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=optimizer,
        device=device,
        num_classes=10,
    )

    start_time = time.time()

    try:
        clf.train(dataloader=train_dl, max_iter=max_iter, verbose=False)
    except:
        print("Bad configuration")
        return None

    training_time = time.time() - start_time

    train_metrics = clf.evaluate(dataloader=train_dl)
    test_metrics = clf.evaluate(dataloader=test_dl)

    print(
        f"train acc: {train_metrics['accuracy']:.4f} | "
        f"test acc: {test_metrics['accuracy']:.4f} | "
        f"test F1: {test_metrics['f1_score']:.4f} | "
        f"training time: {training_time:.4f} s"
    )

    name = _config_name(
        output_channels,
        kernel_size,
        use_adamw=use_adamw,
        batch_norm=batch_norm,
        dropout=dropout,
    )
    HISTORIES[name] = clf.history

    return {
        "output_channels": tuple(output_channels),
        "kernel_size": kernel_size,
        "train_accuracy": train_metrics["accuracy"],
        "test_accuracy": test_metrics["accuracy"],
        "test_f1": test_metrics["f1_score"],
        "training_time": round(training_time, 4),
    }


In [ ]:
def run_conv_grid(
    output_channel_configs: list[int],
    kernel_sizes: list[int],
    max_iter: int,
    use_adamw: bool = False,
    batch_norm: bool = False,
    dropout: bool = False,
) -> pd.DataFrame:

    results: list[dict] = []
    for output_channels, kernel_size in product(
        output_channel_configs, kernel_sizes
    ):
        row = train_conv_config(
            output_channels=output_channels,
            kernel_size=kernel_size,
            max_iter=max_iter,
            use_adamw=use_adamw,
            batch_norm=batch_norm,
            dropout=dropout,
        )
        if row is not None:
            results.append(row)

    return pd.DataFrame(results).sort_values(
        by="test_accuracy", ascending=False
    )

In [ ]:
results_df = run_conv_grid(
    output_channel_configs=OUTPUT_CHANNEL_CONFIGS,
    kernel_sizes=KERNEL_SIZES,
    max_iter=MAX_ITER,
)
results_df


## Dropout


In [ ]:
results_df = run_conv_grid(
    output_channel_configs=SHALLOW_OUTPUT_CHANNEL_CONFIGS,
    kernel_sizes=[3],
    max_iter=MAX_ITER,
    dropout=True,
)
results_df


## AdamW

In [ ]:
results_df = run_conv_grid(
    output_channel_configs=SHALLOW_OUTPUT_CHANNEL_CONFIGS,
    kernel_sizes=[3],
    max_iter=MAX_ITER,
    use_adamw=True,
)
results_df


## Batch Normalization

In [ ]:
results_df = run_conv_grid(
    output_channel_configs=SHALLOW_OUTPUT_CHANNEL_CONFIGS,
    kernel_sizes=[3],
    max_iter=MAX_ITER,
    batch_norm=True,
)
results_df


## Combined

In [ ]:
results_df = run_conv_grid(
    output_channel_configs=SHALLOW_OUTPUT_CHANNEL_CONFIGS,
    kernel_sizes=[3],
    max_iter=MAX_ITER,
    use_adamw=True,
    batch_norm=True,
    dropout=True,
)
results_df


In [ ]:
from helpers.training_history import plot_compare

plot_compare(
    ("without dropout", HISTORIES["out=[16], k=3"]),
    ("with dropout", HISTORIES["out=[16], k=3, dropout"]),
)
plt.plot()

In [ ]:
plot_compare(
    ("without dropout", HISTORIES["out=[32, 64, 128, 256], k=3"]),
    ("with dropout", HISTORIES["out=[32, 64, 128, 256], k=3, dropout"]),
)
plt.plot()

In [ ]:
plot_compare(
    ("adam", HISTORIES["out=[16, 32], k=3"]),
    ("adamw", HISTORIES["out=[16, 32], k=3, adamw"]),
)
plt.plot()

In [ ]:
plot_compare(
    ("adam", HISTORIES["out=[16], k=3"]),
    ("adamw", HISTORIES["out=[16], k=3, adamw"]),
)
plt.plot()

In [ ]:
plot_compare(
    ("batch norm", HISTORIES["out=[16], k=3"]),
    ("no batch norm", HISTORIES["out=[16], k=3, bn"]),
)
plt.plot()

In [ ]:
plot_compare(
    ("batch norm", HISTORIES["out=[32, 64, 128, 256], k=3"]),
    ("no batch norm", HISTORIES["out=[32, 64, 128, 256], k=3, bn"]),
)
plt.plot()

In [ ]:
plot_compare(
    ("base", HISTORIES["out=[32, 64, 128, 256], k=3"]),
    ("full", HISTORIES["out=[32, 64, 128, 256], k=3, adamw+bn+dropout"]),
)
plt.plot()